In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect("cricket.db")
cursor = conn.cursor()

print("Database created successfully")

Database created successfully


In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS players (
    player_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT,
    country TEXT,
    playing_role TEXT,
    batting_style TEXT,
    bowling_style TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS teams (
    team_id INTEGER PRIMARY KEY AUTOINCREMENT,
    team_name TEXT,
    country TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS venues (
    venue_id INTEGER PRIMARY KEY AUTOINCREMENT,
    venue_name TEXT,
    city TEXT,
    country TEXT,
    capacity INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS matches (
    match_id INTEGER PRIMARY KEY AUTOINCREMENT,
    match_description TEXT,
    team1 TEXT,
    team2 TEXT,
    winning_team TEXT,
    venue_name TEXT,
    match_date TEXT,
    match_format TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS batting_stats (
    stat_id INTEGER PRIMARY KEY AUTOINCREMENT,
    player_name TEXT,
    match_format TEXT,
    runs INTEGER,
    balls INTEGER,
    fours INTEGER,
    sixes INTEGER,
    strike_rate REAL,
    centuries INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS bowling_stats (
    stat_id INTEGER PRIMARY KEY AUTOINCREMENT,
    player_name TEXT,
    match_format TEXT,
    overs REAL,
    runs_given INTEGER,
    wickets INTEGER,
    economy REAL
)
""")

conn.commit()

print("Tables created successfully")

Tables created successfully


In [4]:
players_data = [
    ("Virat Kohli", "India", "Batsman", "Right-hand bat", "Right-arm medium"),
    ("Rohit Sharma", "India", "Batsman", "Right-hand bat", "Right-arm offbreak"),
    ("Jasprit Bumrah", "India", "Bowler", "Right-hand bat", "Right-arm fast"),
    ("Ben Stokes", "England", "All-rounder", "Left-hand bat", "Right-arm fast-medium"),
    ("Babar Azam", "Pakistan", "Batsman", "Right-hand bat", "Right-arm offbreak")
]

cursor.executemany("""
INSERT INTO players 
(full_name, country, playing_role, batting_style, bowling_style)
VALUES (?, ?, ?, ?, ?)
""", players_data)

venues_data = [
    ("Eden Gardens", "Kolkata", "India", 68000),
    ("Wankhede Stadium", "Mumbai", "India", 33000),
    ("Lords", "London", "England", 31000),
    ("MCG", "Melbourne", "Australia", 100000)
]

cursor.executemany("""
INSERT INTO venues 
(venue_name, city, country, capacity)
VALUES (?, ?, ?, ?)
""", venues_data)

matches_data = [
    ("India vs Australia 1st ODI", "India", "Australia", "India", "Wankhede Stadium", "2024-01-15", "ODI"),
    ("India vs England Test", "India", "England", "England", "Eden Gardens", "2024-02-10", "Test"),
    ("Pakistan vs England T20", "Pakistan", "England", "Pakistan", "Lords", "2024-03-20", "T20I")
]

cursor.executemany("""
INSERT INTO matches 
(match_description, team1, team2, winning_team, venue_name, match_date, match_format)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", matches_data)

batting_data = [
    ("Virat Kohli", "ODI", 13848, 14797, 1294, 152, 93.58, 50),
    ("Rohit Sharma", "ODI", 10709, 11600, 994, 323, 92.31, 31),
    ("Babar Azam", "ODI", 5729, 6300, 520, 60, 90.93, 19),
    ("Ben Stokes", "ODI", 3463, 3900, 300, 90, 88.79, 5)
]

cursor.executemany("""
INSERT INTO batting_stats 
(player_name, match_format, runs, balls, fours, sixes, strike_rate, centuries)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", batting_data)

bowling_data = [
    ("Jasprit Bumrah", "ODI", 1400.2, 6200, 149, 4.42),
    ("Ben Stokes", "ODI", 500.4, 2800, 74, 5.59)
]

cursor.executemany("""
INSERT INTO bowling_stats 
(player_name, match_format, overs, runs_given, wickets, economy)
VALUES (?, ?, ?, ?, ?, ?)
""", bowling_data)

conn.commit()

print("Sample data inserted successfully")

Sample data inserted successfully


In [5]:
query = """
SELECT * FROM players
"""

pd.read_sql_query(query, conn)

,player_id,full_name,country,playing_role,batting_style,bowling_style
0,1,Virat Kohli,India,Batsman,Right-hand bat,Right-arm medium
1,2,Rohit Sharma,India,Batsman,Right-hand bat,Right-arm offbreak
2,3,Jasprit Bumrah,India,Bowler,Right-hand bat,Right-arm fast
3,4,Ben Stokes,England,All-rounder,Left-hand bat,Right-arm fast-medium
4,5,Babar Azam,Pakistan,Batsman,Right-hand bat,Right-arm offbreak


In [6]:
query = """
SELECT full_name, playing_role, batting_style, bowling_style
FROM players
WHERE country = 'India'
"""

pd.read_sql_query(query, conn)

,full_name,playing_role,batting_style,bowling_style
0,Virat Kohli,Batsman,Right-hand bat,Right-arm medium
1,Rohit Sharma,Batsman,Right-hand bat,Right-arm offbreak
2,Jasprit Bumrah,Bowler,Right-hand bat,Right-arm fast


In [7]:
query = """
SELECT player_name, runs, 
       ROUND(runs * 1.0 / NULLIF(centuries, 0), 2) AS runs_per_century,
       centuries
FROM batting_stats
WHERE match_format = 'ODI'
ORDER BY runs DESC
LIMIT 10
"""

pd.read_sql_query(query, conn)

,player_name,runs,runs_per_century,centuries
0,Virat Kohli,13848,276.96,50
1,Rohit Sharma,10709,345.45,31
2,Babar Azam,5729,301.53,19
3,Ben Stokes,3463,692.60,5


In [8]:
query = """
SELECT
    m.match_description,
    m.team1,
    m.team2,
    m.venue_name,
    v.city,
    m.match_date
FROM matches m
JOIN venues v
ON m.venue_name = v.venue_name
ORDER BY m.match_date DESC
"""

pd.read_sql_query(query, conn)

,match_description,team1,team2,venue_name,city,match_date
0,Pakistan vs England T20,Pakistan,England,Lords,London,2024-03-20
1,India vs England Test,India,England,Eden Gardens,Kolkata,2024-02-10
2,India vs Australia 1st ODI,India,Australia,Wankhede Stadium,Mumbai,2024-01-15


In [9]:
query = """
SELECT
    player_name,
    runs AS total_runs,
    ROUND(runs * 1.0 / NULLIF(centuries, 0), 2) AS sample_average,
    centuries
FROM batting_stats
WHERE match_format = 'ODI'
ORDER BY runs DESC
LIMIT 10
"""

pd.read_sql_query(query, conn)

,player_name,total_runs,sample_average,centuries
0,Virat Kohli,13848,276.96,50
1,Rohit Sharma,10709,345.45,31
2,Babar Azam,5729,301.53,19
3,Ben Stokes,3463,692.60,5


In [10]:
query = """
SELECT
    venue_name,
    city,
    country,
    capacity
FROM venues
WHERE capacity > 25000
ORDER BY capacity DESC
LIMIT 10
"""

pd.read_sql_query(query, conn)

,venue_name,city,country,capacity
0,MCG,Melbourne,Australia,100000
1,Eden Gardens,Kolkata,India,68000
2,Wankhede Stadium,Mumbai,India,33000
3,Lords,London,England,31000


In [11]:
query = """
SELECT
    winning_team AS team_name,
    COUNT(*) AS total_wins
FROM matches
GROUP BY winning_team
ORDER BY total_wins DESC
"""

pd.read_sql_query(query, conn)

,team_name,total_wins
0,Pakistan,1
1,India,1
2,England,1


In [12]:
query = """
SELECT
    playing_role,
    COUNT(*) AS total_players
FROM players
GROUP BY playing_role
ORDER BY total_players DESC
"""

pd.read_sql_query(query, conn)

,playing_role,total_players
0,Batsman,3
1,Bowler,1
2,All-rounder,1


In [13]:
cursor.execute("""
ALTER TABLE batting_stats
ADD COLUMN highest_score INTEGER
""")

conn.commit()

In [14]:
cursor.execute("UPDATE batting_stats SET highest_score = 183 WHERE player_name = 'Virat Kohli'")
cursor.execute("UPDATE batting_stats SET highest_score = 264 WHERE player_name = 'Rohit Sharma'")
cursor.execute("UPDATE batting_stats SET highest_score = 158 WHERE player_name = 'Babar Azam'")
cursor.execute("UPDATE batting_stats SET highest_score = 102 WHERE player_name = 'Ben Stokes'")

conn.commit()

In [15]:
query = """
SELECT
    match_format,
    MAX(highest_score) AS highest_individual_score
FROM batting_stats
GROUP BY match_format
"""

pd.read_sql_query(query, conn)

,match_format,highest_individual_score
0,ODI,264


In [16]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS series (
    series_id INTEGER PRIMARY KEY AUTOINCREMENT,
    series_name TEXT,
    host_country TEXT,
    match_type TEXT,
    start_date TEXT,
    total_matches INTEGER
)
""")

conn.commit()

print("Series table created")

Series table created


In [17]:
series_data = [
    ("India vs England Series", "India", "Test", "2024-01-25", 5),
    ("Asia Cup", "Sri Lanka", "ODI", "2024-08-20", 15),
    ("T20 World Cup", "USA", "T20I", "2024-06-01", 55),
    ("Border Gavaskar Trophy", "Australia", "Test", "2023-12-15", 5)
]

cursor.executemany("""
INSERT INTO series
(series_name, host_country, match_type, start_date, total_matches)
VALUES (?, ?, ?, ?, ?)
""", series_data)

conn.commit()

print("Sample data inserted")

Sample data inserted


In [18]:
query = """
SELECT
    series_name,
    host_country,
    match_type,
    start_date,
    total_matches
FROM series
WHERE strftime('%Y', start_date) = '2024'
ORDER BY start_date
"""

pd.read_sql_query(query, conn)

,series_name,host_country,match_type,start_date,total_matches
0,India vs England Series,India,Test,2024-01-25,5
1,T20 World Cup,USA,T20I,2024-06-01,55
2,Asia Cup,Sri Lanka,ODI,2024-08-20,15


In [19]:
query = """
SELECT
    p.full_name,
    b.runs AS total_runs,
    bw.wickets AS total_wickets,
    b.match_format
FROM players p
JOIN batting_stats b
ON p.full_name = b.player_name
JOIN bowling_stats bw
ON p.full_name = bw.player_name
WHERE p.playing_role = 'All-rounder'
  AND b.runs > 1000
  AND bw.wickets > 50
"""

pd.read_sql_query(query, conn)

,full_name,total_runs,total_wickets,match_format
0,Ben Stokes,3463,74,ODI


In [20]:
query = """
SELECT
    match_description,
    team1,
    team2,
    winning_team,
    venue_name,
    match_date
FROM matches
ORDER BY match_date DESC
LIMIT 20
"""

pd.read_sql_query(query, conn)

,match_description,team1,team2,winning_team,venue_name,match_date
0,Pakistan vs England T20,Pakistan,England,Pakistan,Lords,2024-03-20
1,India vs England Test,India,England,England,Eden Gardens,2024-02-10
2,India vs Australia 1st ODI,India,Australia,India,Wankhede Stadium,2024-01-15


In [21]:
extra_batting = [
("Virat Kohli","Test",9230,16000,1000,30,55.0,29,254),
("Virat Kohli","T20I",4188,3000,350,120,137.0,1,122),

("Rohit Sharma","Test",4137,7500,450,80,46.0,12,212),
("Rohit Sharma","T20I",4231,3100,380,205,140.0,5,118)
]

cursor.executemany("""
INSERT INTO batting_stats
(player_name,match_format,runs,balls,fours,sixes,strike_rate,centuries,highest_score)
VALUES (?,?,?,?,?,?,?,?,?)
""", extra_batting)

conn.commit()

In [22]:
query = """
SELECT
    player_name,

    SUM(CASE
        WHEN match_format='Test'
        THEN runs ELSE 0 END) AS Test_runs,

    SUM(CASE
        WHEN match_format='ODI'
        THEN runs ELSE 0 END) AS ODI_runs,

    SUM(CASE
        WHEN match_format='T20I'
        THEN runs ELSE 0 END) AS T20I_runs,

    ROUND(AVG(strike_rate),2) AS overall_avg_sr

FROM batting_stats

GROUP BY player_name

HAVING COUNT(DISTINCT match_format)>=2
"""

pd.read_sql_query(query, conn)

,player_name,Test_runs,ODI_runs,T20I_runs,overall_avg_sr
0,Rohit Sharma,4137,10709,4231,92.77
1,Virat Kohli,9230,13848,4188,95.19


In [23]:
query = """
SELECT
    m.winning_team AS team_name,
    CASE 
        WHEN m.winning_team = v.country THEN 'Home Win'
        ELSE 'Away Win'
    END AS match_condition,
    COUNT(*) AS total_wins
FROM matches m
JOIN venues v
ON m.venue_name = v.venue_name
GROUP BY m.winning_team, match_condition
ORDER BY team_name
"""

pd.read_sql_query(query, conn)

,team_name,match_condition,total_wins
0,England,Away Win,1
1,India,Home Win,1
2,Pakistan,Away Win,1


In [24]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS partnerships (
    partnership_id INTEGER PRIMARY KEY AUTOINCREMENT,
    match_id INTEGER,
    innings_number INTEGER,
    player1 TEXT,
    player2 TEXT,
    batting_position1 INTEGER,
    batting_position2 INTEGER,
    partnership_runs INTEGER
)
""")

conn.commit()

In [25]:
query = """
SELECT
    player1,
    player2,
    partnership_runs,
    innings_number
FROM partnerships
WHERE ABS(batting_position1 - batting_position2) = 1
  AND partnership_runs >= 100
ORDER BY partnership_runs DESC
"""

pd.read_sql_query(query, conn)

,player1,player2,partnership_runs,innings_number


In [26]:
pd.read_sql_query("SELECT * FROM partnerships", conn)

,partnership_id,match_id,innings_number,player1,player2,batting_position1,batting_position2,partnership_runs


In [27]:
partnership_data = [
    (1, 1, "Virat Kohli", "Rohit Sharma", 2, 3, 145),
    (1, 1, "Rohit Sharma", "KL Rahul", 3, 4, 80),
    (2, 1, "Ben Stokes", "Joe Root", 5, 6, 120),
    (3, 1, "Babar Azam", "Mohammad Rizwan", 1, 2, 105)
]

cursor.executemany("""
INSERT INTO partnerships
(match_id, innings_number, player1, player2, batting_position1, batting_position2, partnership_runs)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", partnership_data)

conn.commit()

In [28]:
query = """
SELECT
    player1,
    player2,
    partnership_runs,
    innings_number
FROM partnerships
WHERE ABS(batting_position1 - batting_position2) = 1
  AND partnership_runs >= 100
ORDER BY partnership_runs DESC
"""

pd.read_sql_query(query, conn)

,player1,player2,partnership_runs,innings_number
0,Virat Kohli,Rohit Sharma,145,1
1,Ben Stokes,Joe Root,120,1
2,Babar Azam,Mohammad Rizwan,105,1


In [29]:
query = """
SELECT
    bw.player_name,
    m.venue_name,
    ROUND(AVG(bw.economy), 2) AS average_economy,
    SUM(bw.wickets) AS total_wickets,
    COUNT(*) AS matches_played
FROM bowling_stats bw
JOIN matches m
ON bw.match_format = m.match_format
WHERE bw.overs >= 4
GROUP BY bw.player_name, m.venue_name
ORDER BY total_wickets DESC
"""

pd.read_sql_query(query, conn)

,player_name,venue_name,average_economy,total_wickets,matches_played
0,Jasprit Bumrah,Wankhede Stadium,4.42,149,1
1,Ben Stokes,Wankhede Stadium,5.59,74,1


In [30]:
cursor.execute("""
ALTER TABLE matches
ADD COLUMN victory_margin INTEGER
""")

cursor.execute("""
ALTER TABLE matches
ADD COLUMN victory_type TEXT
""")

conn.commit()

In [31]:
cursor.execute("""
UPDATE matches
SET victory_margin = 3,
    victory_type = 'wickets'
WHERE match_id = 1
""")

cursor.execute("""
UPDATE matches
SET victory_margin = 40,
    victory_type = 'runs'
WHERE match_id = 2
""")

cursor.execute("""
UPDATE matches
SET victory_margin = 7,
    victory_type = 'wickets'
WHERE match_id = 3
""")

conn.commit()

In [32]:
query = """
SELECT
    b.player_name,
    ROUND(AVG(b.runs),2) AS avg_runs,
    COUNT(*) AS close_matches_played

FROM batting_stats b
JOIN matches m
ON b.match_format = m.match_format

WHERE
(
m.victory_type='runs'
AND m.victory_margin <50
)

OR

(
m.victory_type='wickets'
AND m.victory_margin <5
)

GROUP BY b.player_name
ORDER BY avg_runs DESC
"""

pd.read_sql_query(query, conn)

,player_name,avg_runs,close_matches_played
0,Virat Kohli,11539.0,2
1,Rohit Sharma,7423.0,2
2,Babar Azam,5729.0,1
3,Ben Stokes,3463.0,1


In [33]:
cursor.execute("""
ALTER TABLE batting_stats
ADD COLUMN match_year INTEGER
""")

conn.commit()

In [34]:
cursor.execute("UPDATE batting_stats SET match_year = 2024 WHERE player_name IN ('Virat Kohli', 'Rohit Sharma')")
cursor.execute("UPDATE batting_stats SET match_year = 2023 WHERE player_name IN ('Babar Azam', 'Ben Stokes')")

conn.commit()

In [35]:
query = """
SELECT
    player_name,
    match_year,
    ROUND(AVG(runs), 2) AS average_runs,
    ROUND(AVG(strike_rate), 2) AS average_strike_rate,
    COUNT(*) AS matches_played
FROM batting_stats
WHERE match_year >= 2020
GROUP BY player_name, match_year
ORDER BY match_year DESC, average_runs DESC
"""

pd.read_sql_query(query, conn)

,player_name,match_year,average_runs,average_strike_rate,matches_played
0,Virat Kohli,2024,9088.67,95.19,3
1,Rohit Sharma,2024,6359.00,92.77,3
2,Babar Azam,2023,5729.00,90.93,1
3,Ben Stokes,2023,3463.00,88.79,1


In [36]:
cursor.execute("""
ALTER TABLE matches
ADD COLUMN toss_winner TEXT
""")

cursor.execute("""
ALTER TABLE matches
ADD COLUMN toss_decision TEXT
""")

conn.commit()

In [37]:
cursor.execute("""
UPDATE matches
SET toss_winner = 'India',
    toss_decision = 'bat'
WHERE match_id = 1
""")

cursor.execute("""
UPDATE matches
SET toss_winner = 'India',
    toss_decision = 'bowl'
WHERE match_id = 2
""")

cursor.execute("""
UPDATE matches
SET toss_winner = 'Pakistan',
    toss_decision = 'bat'
WHERE match_id = 3
""")

conn.commit()

In [38]:
query = """
SELECT
    toss_decision,
    COUNT(*) AS total_matches,
    SUM(CASE 
        WHEN toss_winner = winning_team THEN 1 
        ELSE 0 
    END) AS toss_winner_wins,
    ROUND(
        SUM(CASE 
            WHEN toss_winner = winning_team THEN 1 
            ELSE 0 
        END) * 100.0 / COUNT(*), 
        2
    ) AS win_percentage
FROM matches
GROUP BY toss_decision
"""

pd.read_sql_query(query, conn)

,toss_decision,total_matches,toss_winner_wins,win_percentage
0,bat,2,2,100.0
1,bowl,1,0,0.0


In [39]:
query = """
SELECT
    player_name,
    match_format,
    ROUND(AVG(economy), 2) AS overall_economy,
    SUM(wickets) AS total_wickets,
    COUNT(*) AS matches_bowled,
    ROUND(AVG(overs), 2) AS avg_overs
FROM bowling_stats
WHERE match_format IN ('ODI', 'T20I')
GROUP BY player_name, match_format
HAVING COUNT(*) >= 1
   AND AVG(overs) >= 2
ORDER BY overall_economy ASC
"""

pd.read_sql_query(query, conn)

,player_name,match_format,overall_economy,total_wickets,matches_bowled,avg_overs
0,Jasprit Bumrah,ODI,4.42,149,1,1400.2
1,Ben Stokes,ODI,5.59,74,1,500.4


In [40]:
query = """
SELECT
    player_name,
    ROUND(AVG(runs),2) AS average_runs,
    COUNT(*) AS innings_played,
    ROUND(AVG(strike_rate),2) AS avg_strike_rate
FROM batting_stats
WHERE match_year >= 2022
GROUP BY player_name
HAVING COUNT(*) >= 1
ORDER BY average_runs DESC
"""

pd.read_sql_query(query, conn)

,player_name,average_runs,innings_played,avg_strike_rate
0,Virat Kohli,9088.67,3,95.19
1,Rohit Sharma,6359.00,3,92.77
2,Babar Azam,5729.00,1,90.93
3,Ben Stokes,3463.00,1,88.79


In [41]:
query = """
SELECT
    player_name,

SUM(CASE
WHEN match_format='Test'
THEN 1 ELSE 0 END)
AS Test_matches,

SUM(CASE
WHEN match_format='ODI'
THEN 1 ELSE 0 END)
AS ODI_matches,

SUM(CASE
WHEN match_format='T20I'
THEN 1 ELSE 0 END)
AS T20_matches,

ROUND(AVG(runs),2)
AS overall_avg_runs

FROM batting_stats

GROUP BY player_name

HAVING COUNT(*) >=1
"""

pd.read_sql_query(query, conn)

,player_name,Test_matches,ODI_matches,T20_matches,overall_avg_runs
0,Babar Azam,0,1,0,5729.00
1,Ben Stokes,0,1,0,3463.00
2,Rohit Sharma,1,1,1,6359.00
3,Virat Kohli,1,1,1,9088.67


In [42]:
query = """
SELECT

b.player_name,

b.match_format,

ROUND(
(b.runs * 0.01)
+
(b.strike_rate * 0.3)
+
(COALESCE(bw.wickets,0) * 2)
+
((6 - COALESCE(bw.economy,6))*2)

,2)

AS total_score

FROM batting_stats b

LEFT JOIN bowling_stats bw
ON b.player_name = bw.player_name
AND b.match_format = bw.match_format

ORDER BY total_score DESC
"""

pd.read_sql_query(query, conn)

,player_name,match_format,total_score
0,Ben Stokes,ODI,210.09
1,Virat Kohli,ODI,166.55
2,Rohit Sharma,ODI,134.78
3,Virat Kohli,Test,108.80
4,Babar Azam,ODI,84.57
5,Rohit Sharma,T20I,84.31
6,Virat Kohli,T20I,82.98
7,Rohit Sharma,Test,55.17


In [43]:
query = """
SELECT
    team1,
    team2,
    COUNT(*) AS total_matches,
    SUM(CASE WHEN winning_team = team1 THEN 1 ELSE 0 END) AS team1_wins,
    SUM(CASE WHEN winning_team = team2 THEN 1 ELSE 0 END) AS team2_wins,
    ROUND(
        SUM(CASE WHEN winning_team = team1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS team1_win_percentage,
    ROUND(
        SUM(CASE WHEN winning_team = team2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS team2_win_percentage
FROM matches
GROUP BY team1, team2
ORDER BY total_matches DESC
"""

pd.read_sql_query(query, conn)

,team1,team2,total_matches,team1_wins,team2_wins,team1_win_percentage,team2_win_percentage
0,India,Australia,1,1,0,100.0,0.0
1,India,England,1,0,1,0.0,100.0
2,Pakistan,England,1,1,0,100.0,0.0


In [44]:
query = """
SELECT
    player_name,

    ROUND(AVG(runs),2) AS avg_runs,

    ROUND(AVG(strike_rate),2) AS avg_sr,

    SUM(
        CASE
        WHEN runs >=50
        THEN 1
        ELSE 0
        END
    ) AS fifty_plus_scores,

    CASE

    WHEN AVG(runs)>=100
    THEN 'Excellent Form'

    WHEN AVG(runs)>=50
    THEN 'Good Form'

    WHEN AVG(runs)>=30
    THEN 'Average Form'

    ELSE 'Poor Form'

    END AS form_category

FROM batting_stats

GROUP BY player_name

ORDER BY avg_runs DESC
"""

pd.read_sql_query(query, conn)

,player_name,avg_runs,avg_sr,fifty_plus_scores,form_category
0,Virat Kohli,9088.67,95.19,3,Excellent Form
1,Rohit Sharma,6359.00,92.77,3,Excellent Form
2,Babar Azam,5729.00,90.93,1,Excellent Form
3,Ben Stokes,3463.00,88.79,1,Excellent Form


In [45]:
query = """
SELECT

player1,
player2,

COUNT(*) AS total_partnerships,

ROUND(AVG(partnership_runs),2)
AS avg_partnership_runs,

MAX(partnership_runs)
AS highest_partnership,

SUM(
CASE
WHEN partnership_runs >50
THEN 1
ELSE 0
END
)

AS successful_partnerships

FROM partnerships

GROUP BY player1,player2

HAVING COUNT(*)>=1

ORDER BY avg_partnership_runs DESC
"""

pd.read_sql_query(query, conn)

,player1,player2,total_partnerships,avg_partnership_runs,highest_partnership,successful_partnerships
0,Virat Kohli,Rohit Sharma,1,145.0,145,1
1,Ben Stokes,Joe Root,1,120.0,120,1
2,Babar Azam,Mohammad Rizwan,1,105.0,105,1
3,Rohit Sharma,KL Rahul,1,80.0,80,1


In [46]:
query = """
SELECT
    player_name,
    match_year,
    ROUND(AVG(runs),2) AS avg_runs,
    ROUND(AVG(strike_rate),2) AS avg_strike_rate,

    CASE
        WHEN AVG(runs) >= 100 THEN 'Career Ascending'
        WHEN AVG(runs) >= 50 THEN 'Career Stable'
        ELSE 'Career Declining'
    END AS career_phase

FROM batting_stats

WHERE match_year IS NOT NULL

GROUP BY player_name, match_year

ORDER BY player_name, match_year
"""

pd.read_sql_query(query, conn)

,player_name,match_year,avg_runs,avg_strike_rate,career_phase
0,Babar Azam,2023,5729.00,90.93,Career Ascending
1,Ben Stokes,2023,3463.00,88.79,Career Ascending
2,Rohit Sharma,2024,6359.00,92.77,Career Ascending
3,Virat Kohli,2024,9088.67,95.19,Career Ascending


In [2]:
import requests
import pandas as pd

url = "https://free-cricbuzz-cricket-api.p.rapidapi.com/cricket-match-info?matchid=102040"

headers = {
    "X-RapidAPI-Key": "c6ddfd3c50mshf4f1482c7364924p1a050bjsn31619468ec55",
    "X-RapidAPI-Host": "free-cricbuzz-cricket-api.p.rapidapi.com"
}

response = requests.get(url, headers=headers)

print(response.status_code)
print(response.text[:500])

200
{"status":"success","response":{"matchInfo":{}}}


In [3]:
url = "https://free-cricbuzz-cricket-api.p.rapidapi.com/cricket-schedule"

response = requests.get(url, headers=headers)

print(response.status_code)
print(response.text[:1000])

200
{"status":"success","response":{"schedules":[{"scheduleAdWrapper":{"date":"MON, MAY 25 2026","matchScheduleList":[{"seriesName":"New Zealand A Women tour of Sri Lanka, 2026","matchInfo":[{"matchId":155921,"seriesId":12239,"matchDesc":"3rd Unofficial T20I","matchFormat":"T20","startDate":"1779683400000","endDate":"1779753599000","team1":{"teamId":1556,"teamName":"Sri Lanka A Women","teamSName":"SLWA","imageId":776273},"team2":{"teamId":3223,"teamName":"New Zealand A Women","teamSName":"NZWA","imageId":971001},"venueInfo":{"ground":"Rangiri Dambulla International Stadium","city":"Dambulla","country":"Sri Lanka","timezone":"+05:30"}}],"seriesId":12239,"seriesCategory":"Women"},{"seriesName":"ICC Men's T20 World Cup Sub Regional Africa Qualifier Group A 2026","matchInfo":[{"matchId":158147,"seriesId":12356,"matchDesc":"8th Match","matchFormat":"T20","startDate":"1779694200000","endDate":"1779753599000","team1":{"teamId":529,"teamName":"Botswana","teamSName":"BW","imageId":776225},"team

In [4]:
fixtures_data = response.json()
fixtures_data

{'status': 'success',
 'response': {'schedules': [{'scheduleAdWrapper': {'date': 'MON, MAY 25 2026',
     'matchScheduleList': [{'seriesName': 'New Zealand A Women tour of Sri Lanka, 2026',
       'matchInfo': [{'matchId': 155921,
         'seriesId': 12239,
         'matchDesc': '3rd Unofficial T20I',
         'matchFormat': 'T20',
         'startDate': '1779683400000',
         'endDate': '1779753599000',
         'team1': {'teamId': 1556,
          'teamName': 'Sri Lanka A Women',
          'teamSName': 'SLWA',
          'imageId': 776273},
         'team2': {'teamId': 3223,
          'teamName': 'New Zealand A Women',
          'teamSName': 'NZWA',
          'imageId': 971001},
         'venueInfo': {'ground': 'Rangiri Dambulla International Stadium',
          'city': 'Dambulla',
          'country': 'Sri Lanka',
          'timezone': '+05:30'}}],
       'seriesId': 12239,
       'seriesCategory': 'Women'},
      {'seriesName': "ICC Men's T20 World Cup Sub Regional Africa Qualifie

In [5]:
df_fixtures = pd.json_normalize(fixtures_data)
df_fixtures.head()

,status,response.schedules
0,success,"[{'scheduleAdWrapper': {'date': 'MON, MAY 25 2..."


In [7]:
matches = []

for item in fixtures_data["response"]["schedules"]:
    schedule = item["scheduleAdWrapper"]
    date = schedule.get("date")

    for match in schedule.get("matchInfo", []):
        matches.append({
            "date": date,
            "match_id": match.get("matchId"),
            "series_name": match.get("seriesName"),
            "match_desc": match.get("matchDesc"),
            "match_format": match.get("matchFormat"),
            "start_date": match.get("startDate"),
            "end_date": match.get("endDate"),
            "team1": match.get("team1", {}).get("teamName"),
            "team2": match.get("team2", {}).get("teamName"),
            "venue": match.get("venueInfo", {}).get("ground"),
            "city": match.get("venueInfo", {}).get("city"),
            "country": match.get("venueInfo", {}).get("country")
        })

df_fixtures_clean = pd.DataFrame(matches)

df_fixtures_clean.head()

""


In [8]:
fixtures_data["response"]["schedules"][0]

{'scheduleAdWrapper': {'date': 'MON, MAY 25 2026',
  'matchScheduleList': [{'seriesName': 'New Zealand A Women tour of Sri Lanka, 2026',
    'matchInfo': [{'matchId': 155921,
      'seriesId': 12239,
      'matchDesc': '3rd Unofficial T20I',
      'matchFormat': 'T20',
      'startDate': '1779683400000',
      'endDate': '1779753599000',
      'team1': {'teamId': 1556,
       'teamName': 'Sri Lanka A Women',
       'teamSName': 'SLWA',
       'imageId': 776273},
      'team2': {'teamId': 3223,
       'teamName': 'New Zealand A Women',
       'teamSName': 'NZWA',
       'imageId': 971001},
      'venueInfo': {'ground': 'Rangiri Dambulla International Stadium',
       'city': 'Dambulla',
       'country': 'Sri Lanka',
       'timezone': '+05:30'}}],
    'seriesId': 12239,
    'seriesCategory': 'Women'},
   {'seriesName': "ICC Men's T20 World Cup Sub Regional Africa Qualifier Group A 2026",
    'matchInfo': [{'matchId': 158147,
      'seriesId': 12356,
      'matchDesc': '8th Match',
    

In [9]:
fixtures_data["response"]["schedules"][0].keys()

dict_keys(['scheduleAdWrapper'])

In [10]:
matches = []

for item in fixtures_data["response"]["schedules"]:
    wrapper = item["scheduleAdWrapper"]
    date = wrapper.get("date")

    for schedule in wrapper.get("matchScheduleList", []):
        for match in schedule.get("matchInfo", []):
            matches.append({
                "date": date,
                "match_id": match.get("matchId"),
                "series_name": match.get("seriesName"),
                "match_desc": match.get("matchDesc"),
                "match_format": match.get("matchFormat"),
                "start_date": match.get("startDate"),
                "end_date": match.get("endDate"),
                "team1": match.get("team1", {}).get("teamName"),
                "team2": match.get("team2", {}).get("teamName"),
                "venue": match.get("venueInfo", {}).get("ground"),
                "city": match.get("venueInfo", {}).get("city"),
                "country": match.get("venueInfo", {}).get("country")
            })

df_fixtures_clean = pd.DataFrame(matches)

df_fixtures_clean.head()

,date,match_id,series_name,match_desc,match_format,start_date,end_date,team1,team2,venue,city,country
0,"MON, MAY 25 2026",155921,None,3rd Unofficial T20I,T20,1779683400000,1779753599000,Sri Lanka A Women,New Zealand A Women,Rangiri Dambulla International Stadium,Dambulla,Sri Lanka
1,"MON, MAY 25 2026",158147,None,8th Match,T20,1779694200000,1779753599000,Botswana,Cameroon,Botswana Cricket Association Oval 2,Gaborone,Botswana
2,"MON, MAY 25 2026",158143,None,7th Match,T20,1779694200000,1779753599000,Kenya,Rwanda,Botswana Cricket Association Oval 1,Gaborone,Botswana
3,"MON, MAY 25 2026",158158,None,9th Match,T20,1779709800000,1779753599000,Ivory Coast,Mali,Botswana Cricket Association Oval 2,Gaborone,Botswana
4,"MON, MAY 25 2026",157300,None,8th Match,T20,1779701400000,1779753599000,Lancashire Women,The Blaze Women,Emirates Old Trafford,Manchester,England


In [13]:
df_fixtures_clean.to_sql("api_fixtures", conn, if_exists="replace", index=False)

pd.read_sql_query("SELECT * FROM api_fixtures LIMIT 10", conn)

,date,match_id,series_name,match_desc,match_format,start_date,end_date,team1,team2,venue,city,country
0,"MON, MAY 25 2026",155921,None,3rd Unofficial T20I,T20,1779683400000,1779753599000,Sri Lanka A Women,New Zealand A Women,Rangiri Dambulla International Stadium,Dambulla,Sri Lanka
1,"MON, MAY 25 2026",158147,None,8th Match,T20,1779694200000,1779753599000,Botswana,Cameroon,Botswana Cricket Association Oval 2,Gaborone,Botswana
2,"MON, MAY 25 2026",158143,None,7th Match,T20,1779694200000,1779753599000,Kenya,Rwanda,Botswana Cricket Association Oval 1,Gaborone,Botswana
3,"MON, MAY 25 2026",158158,None,9th Match,T20,1779709800000,1779753599000,Ivory Coast,Mali,Botswana Cricket Association Oval 2,Gaborone,Botswana
4,"MON, MAY 25 2026",157300,None,8th Match,T20,1779701400000,1779753599000,Lancashire Women,The Blaze Women,Emirates Old Trafford,Manchester,England
5,"MON, MAY 25 2026",155057,None,"1st unofficial Test, Day 4",TEST,1779703200000,1779753599000,England Lions,South Africa A,Arundel Castle,Arundel,England
6,"MON, MAY 25 2026",157857,None,8th Match,T20,1779715800000,1779753599000,Kent Women,Sussex Women,St Lawrence Ground,Canterbury,England
7,"MON, MAY 25 2026",145971,None,17th Match,T20,1779719400000,1779753599000,Kent,Sussex,St Lawrence Ground,Canterbury,England
8,"MON, MAY 25 2026",145960,None,16th Match,T20,1779721200000,1779753599000,Lancashire,Nottinghamshire,Emirates Old Trafford,Manchester,England
9,"MON, MAY 25 2026",129431,None,3rd T20I,T20,1779730200000,1779753599000,England Women,New Zealand Women,County Ground,Hove,England


In [12]:
import sqlite3

conn = sqlite3.connect("cricket.db")

In [14]:
df_fixtures_clean.to_sql(
    "api_fixtures",
    conn,
    if_exists="replace",
    index=False
)

pd.read_sql_query("SELECT * FROM api_fixtures LIMIT 10", conn)

,date,match_id,series_name,match_desc,match_format,start_date,end_date,team1,team2,venue,city,country
0,"MON, MAY 25 2026",155921,None,3rd Unofficial T20I,T20,1779683400000,1779753599000,Sri Lanka A Women,New Zealand A Women,Rangiri Dambulla International Stadium,Dambulla,Sri Lanka
1,"MON, MAY 25 2026",158147,None,8th Match,T20,1779694200000,1779753599000,Botswana,Cameroon,Botswana Cricket Association Oval 2,Gaborone,Botswana
2,"MON, MAY 25 2026",158143,None,7th Match,T20,1779694200000,1779753599000,Kenya,Rwanda,Botswana Cricket Association Oval 1,Gaborone,Botswana
3,"MON, MAY 25 2026",158158,None,9th Match,T20,1779709800000,1779753599000,Ivory Coast,Mali,Botswana Cricket Association Oval 2,Gaborone,Botswana
4,"MON, MAY 25 2026",157300,None,8th Match,T20,1779701400000,1779753599000,Lancashire Women,The Blaze Women,Emirates Old Trafford,Manchester,England
5,"MON, MAY 25 2026",155057,None,"1st unofficial Test, Day 4",TEST,1779703200000,1779753599000,England Lions,South Africa A,Arundel Castle,Arundel,England
6,"MON, MAY 25 2026",157857,None,8th Match,T20,1779715800000,1779753599000,Kent Women,Sussex Women,St Lawrence Ground,Canterbury,England
7,"MON, MAY 25 2026",145971,None,17th Match,T20,1779719400000,1779753599000,Kent,Sussex,St Lawrence Ground,Canterbury,England
8,"MON, MAY 25 2026",145960,None,16th Match,T20,1779721200000,1779753599000,Lancashire,Nottinghamshire,Emirates Old Trafford,Manchester,England
9,"MON, MAY 25 2026",129431,None,3rd T20I,T20,1779730200000,1779753599000,England Women,New Zealand Women,County Ground,Hove,England
